# run_e12
Phase 3 CPU Track Stage A2 Worker 4 (E12): image-anchored PCA-64 orthogonal Procrustes vs ridge linear maps on midpoint `image_caption` / `caption_mean` `G_res`.

- Run on Colab CPU (`requirements-stage2-cpu.txt`, Python 3.11/3.12, single BLAS/OpenMP thread).
- Edit only Cell 3 (`MODEL`, `PLAN_REF`, optional resume refs, or `PHASE1_ROOT`).
- Publishes only under `PHASE1_ROOT/phase3/{maps,predictions,results}/e12/<MODEL>/`.


In [ ]:
import os, sys, subprocess
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/Research-phase3-e12')
CODE_REF = 'phase3'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
# Visible Stage A2 E12 settings.
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
MODEL = 'qwen'  # 'qwen' or 'gemma'
PLAN_REF = None  # Paste {'path': 'phase3/plans/.../_COMPLETE.json', 'sha256': '...'}
RESUME_MAPS_REF = None
RESUME_PREDICTIONS_REF = None
RESUME_ANALYSIS_REF = None


In [ ]:
import json

if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Phase 3 checkout.')
if not PHASE1_ROOT.is_dir():
    raise RuntimeError(f'Missing mounted PHASE1_ROOT directory: {PHASE1_ROOT}')
if MODEL not in ('qwen', 'gemma'):
    raise RuntimeError(f'Invalid MODEL selector: {MODEL!r}')
if not PLAN_REF:
    raise RuntimeError('Set PLAN_REF in Cell 3 to the published phase3_cpu_plan reference.')

def _arg(ref):
    return json.dumps(ref) if isinstance(ref, dict) else str(ref)

cmd = [
    sys.executable, '-u', '-m', 'src.phase3.run_e12',
    '--phase1-root', str(PHASE1_ROOT),
    '--plan', _arg(PLAN_REF),
    '--model', MODEL,
    '--storage-scope', 'persistent-drive',
]
if RESUME_MAPS_REF:
    cmd.extend(['--resume-maps', _arg(RESUME_MAPS_REF)])
if RESUME_PREDICTIONS_REF:
    cmd.extend(['--resume-predictions', _arg(RESUME_PREDICTIONS_REF)])
if RESUME_ANALYSIS_REF:
    cmd.extend(['--resume-analysis', _arg(RESUME_ANALYSIS_REF)])

proc = subprocess.Popen(cmd, cwd=REPO, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
blockers = []
last_complete = None
for line in proc.stdout:
    print(line, end='', flush=True)
    try:
        payload = json.loads(line)
        if isinstance(payload, dict):
            if payload.get('status') == 'blocked':
                blockers.extend(payload.get('blockers', []))
            if payload.get('event') == 'e12_complete':
                last_complete = payload
    except json.JSONDecodeError:
        pass
code = proc.wait()
if code != 0 or last_complete is None:
    detail = '; '.join(blockers) if blockers else f'exit code {code}'
    raise RuntimeError(f'Phase 3 E12 failed ({code}): {detail}')

print('\n=== AUTHORITATIVE PHASE 3 E12 REFERENCES ===')
print(json.dumps({'maps_ref': last_complete['maps_ref'], 'predictions_ref': last_complete['predictions_ref'], 'analysis_ref': last_complete['analysis_ref']}, indent=2))
